In [ ]:
import subprocess
import json
from bs4 import BeautifulSoup

In [160]:
!curl 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pmc&term=variant%5Btw%5D+AND+acmg%5Btw%5D&retmode=json&retmax=100&usehistory=y&api_key=27f971ac2330c36d95280b361f6d45d00308' -o ids.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1379    0  1379    0     0   1540      0 --:--:-- --:--:-- --:--:--  1540


In [ ]:
with open("ids.json", "r") as f:
    d = json.load(f)

id_list = d["esearchresult"]["idlist"]
querykey = d["esearchresult"]["querykey"]
webenv = d["esearchresult"]["webenv"]

output_file = "fetch.xml"

subprocess.run(["curl", f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&webenv={webenv}&query_key={querykey}&retmax=100", "-o", output_file])

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8314k    0 8314k    0     0   557k      0 --:--:--  0:00:14 --:--:--  781k


CompletedProcess(args=['curl', 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&webenv=MCID_6a21af37ad9922905203fa10&query_key=1&retmax=100', '-o', 'fetch.xml'], returncode=0)

In [ ]:
# 1. Load your XML file
with open(output_file, "r", encoding="utf-8") as file:
    xml_content = file.read()


soup = BeautifulSoup(xml_content, "xml")



for article in soup.find_all("article"):
    if article["article-type"] == "research-article":
        article_name = article.find("article-title").get_text()
    
        for id in article.find_all("article-id"):
            if id["pub-id-type"]=="pmcid":
                id_num = id.get_text()
    
        # 2. Clean up formatting tags and remove citation brackets inside paragraphs
        # We want to unwrap text formatting like <italic> but completely delete <xref> citations
        for paragraph in article.find_all("p"):

            for italic in paragraph.find_all("italic"):
                italic.unwrap()
                
            for xref in paragraph.find_all("xref"):
                xref.decompose()

        # 3. Extract text specifically from the Paper Abstract and Body Section
        paper_text_blocks = []
        paper_text_blocks.append("--- TITLE ---")
        paper_text_blocks.append(article_name)

        # Gather Abstract text if present
        abstract = article.find("abstract")
        if abstract:
            paper_text_blocks.append("--- ABSTRACT ---")
            for p in abstract.find_all("p"):
                paper_text_blocks.append(p.get_text().strip())

        # Gather Main Body text (Introduction, Methods, Results, Discussion, etc.)
        body = article.find("body")
        if body:
            for section in body.find_all("sec"):
                title = section.find("title")
                if title:
                    paper_text_blocks.append(f"\n--- {title.get_text().upper()} ---")
                
                for p in section.find_all("p", recursive=False):
                    paper_text_blocks.append(p.get_text().strip())

        # 4. Join text pieces and save/print the result
        clean_paper_text = "\n".join(paper_text_blocks)

        with open(f"./pubmed_papers/{id_num}.txt", "w", encoding="utf-8") as out_file:
            out_file.write(clean_paper_text)